# Loan Default Prediction – EDA & Modeling

This notebook walks through exploratory analysis, preprocessing, model training and SHAP explanations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate_data import generate_loan_dataset
from src.data.load_data import load_raw_data
from src.features.preprocessing import prepare_xy, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.models.train import train_all_models, load_best_model

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

## 1. Generate / Load Data

In [ ]:
# Generate if not present
raw_path = Path.cwd().parent / "data" / "raw" / "loan_data.csv"
if not raw_path.exists():
    generate_loan_dataset(save_path=raw_path)

df = load_raw_data()
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nDefault rate:", df["default"].mean().round(3))

## 2. Quick EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="default", ax=axes[0, 0])
axes[0, 0].set_title("Target Distribution")

# Note: real dataset uses different feature names; adapt boxplots accordingly
plt.tight_layout()
plt.show()

## 3. Train Models

In [ ]:
results = train_all_models()
pd.DataFrame(results["results"]).T.round(4)

## 4. SHAP (run after training)

In [ ]:
from src.explainability.shap_analysis import compute_shap_values, plot_shap_summary
from src.data.load_data import load_processed_data

artifact = load_best_model()
pipe = artifact["pipeline"]
feature_names = artifact["feature_names"]

test_df = load_processed_data("test.parquet")
X_test, y_test = prepare_xy(test_df)

explainer, shap_vals, X_trans = compute_shap_values(pipe, X_test, max_samples=400)
plot_shap_summary(shap_vals, X_trans, feature_names)